# NUBI VIDEO AI STUDIO — Launcher Colab

Launcher iterativo para a aplicação NUBI. **Execute as células em ordem**. O runtime Colab é temporário; use Google Drive para persistência. A URL de proxy é privada à sessão e temporária.

**Versão:** 0.1.0 · **Ambiente:** Google Colab / notebook local compatível

In [ ]:
import os, sys, platform, subprocess, json, time, shutil, socket, urllib.request, pathlib, getpass
from datetime import datetime
NUBI_VERSION = "0.1.0"
NUBI_REPO = "https://github.com/FelipeGomes070898/NUBI_VIDEO_AI_STUDIO.git"
NUBI_BRANCH = "main"
NUBI_PATH = pathlib.Path("/content/NUBI_VIDEO_AI_STUDIO/main") if "google.colab" in sys.modules else pathlib.Path.cwd()
NUBI_PROCESS = None
print(f"NUBI VIDEO AI STUDIO v{NUBI_VERSION}")
print("Data/hora:", datetime.now().isoformat(timespec="seconds"))
print("Ambiente:", "Google Colab" if "google.colab" in sys.modules else "Jupyter/local")
print("Instruções: execute todas as células; configure Drive e Secret LTX opcionalmente.")

## 2 — Detecção de hardware

A detecção não baixa modelos. Modo híbrido usa GPU se disponível e CPU para tarefas auxiliares. A disponibilidade de GPU no Colab pode variar.

In [ ]:
import psutil, shutil
try:
 import torch
 cuda_ok=torch.cuda.is_available()
 gpu_name=torch.cuda.get_device_name(0) if cuda_ok else "Não disponível"
 vram=round(torch.cuda.get_device_properties(0).total_memory/1024**3,1) if cuda_ok else 0
except Exception:
 cuda_ok=False; gpu_name="Não detectada (torch indisponível)"; vram=0
try:
 import psutil
 ram=round(psutil.virtual_memory().total/1024**3,1)
except Exception: ram="desconhecida"
usage=shutil.disk_usage("/")
print({"cpu":platform.processor() or platform.machine(),"ram_gb":ram,"gpu":gpu_name,"vram_gb":vram,"cuda":cuda_ok,"python":platform.python_version(),"disk_free_gb":round(usage.free/1024**3,1)})
NUBI_MODE = "GPU" if cuda_ok else "CPU"
print("Modo sugerido:", NUBI_MODE, "— HYBRID também é possível se houver GPU.")
print("Planejamento: sem GPU, priorize clipes curtos e baixa resolução; Colab não garante GPU ou VRAM.")

## 3 — Google Drive (opcional)

A montagem permite persistir projetos, banco, saídas e cache. Não guarde credenciais no Drive. Execução local sem Colab ignora a montagem.

In [ ]:
NUBI_DRIVE = None
if "google.colab" in sys.modules:
 try:
  from google.colab import drive
  resposta=input("Montar Google Drive para persistência? [s/N]: ").strip().lower()
  if resposta in ("s","sim","y","yes"):
   drive.mount("/content/drive")
   NUBI_DRIVE=pathlib.Path("/content/drive/MyDrive/NUBI")
   NUBI_DRIVE.mkdir(parents=True,exist_ok=True)
   print("Drive montado. Persistência:",NUBI_DRIVE)
  else: print("Drive não montado; arquivos em /content serão temporários.")
 except Exception as e: print("Drive indisponível:",e)
else: print("Fora do Colab: Drive não montado.")

## 4 — Clonar ou atualizar repositório

Clona o repositório público na branch selecionada ou atualiza apenas código versionado. A pasta `storage/` é preservada pelo `.gitignore`; nunca aplica `reset --hard` nem remove arquivos do usuário. Para repositório privado, configure a autenticação no Colab de forma segura, sem embutir token na URL/célula.

In [ ]:
import subprocess, os, pathlib
NUBI_REPO = input(f"Repositório [{NUBI_REPO}]: ").strip() or NUBI_REPO
NUBI_BRANCH = input(f"Branch [{NUBI_BRANCH}]: ").strip() or NUBI_BRANCH
repo_root=pathlib.Path("/content/NUBI_VIDEO_AI_STUDIO") if "google.colab" in sys.modules else pathlib.Path.cwd()
def run_checked(cmd,cwd=None):
 p=subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
 print(p.stdout[-5000:])
 if p.returncode: raise RuntimeError(f"Falha ({p.returncode}): {' '.join(cmd)}")
 return p.stdout
if "google.colab" in sys.modules:
 if (repo_root/".git").exists():
  run_checked(["git","fetch","origin",NUBI_BRANCH],cwd=repo_root)
  run_checked(["git","checkout",NUBI_BRANCH],cwd=repo_root)
  run_checked(["git","pull","--ff-only","origin",NUBI_BRANCH],cwd=repo_root)
 else:
  run_checked(["git","clone","--branch",NUBI_BRANCH,"--depth","1",NUBI_REPO,str(repo_root)])
 NUBI_PATH=repo_root/"main"
else:
 NUBI_PATH=pathlib.Path.cwd()
print("Projeto:",NUBI_PATH)
if (repo_root/".git").exists(): run_checked(["git","status","--short","--branch"],cwd=repo_root)

## 5 — Dependências do sistema

Instala apenas requisitos Python pinados do projeto. FFmpeg/Node são opcionais para a interface-base; o notebook reporta disponibilidade e evita baixar modelos pesados automaticamente.

In [ ]:
assert (NUBI_PATH/"requirements.txt").exists(), f"Não achei requirements.txt em {NUBI_PATH}"
# Instalação idempotente; pip verifica dependências compatíveis sem reinstalar tudo à força.
run_checked([sys.executable,"-m","pip","install","-r",str(NUBI_PATH/"requirements.txt")])
if not shutil.which("ffmpeg") and "google.colab" in sys.modules:
 print("FFmpeg ausente; instalando o pacote do sistema apenas neste caso.")
 run_checked(["apt-get","update","-qq"])
 run_checked(["apt-get","install","-y","-qq","ffmpeg"])
for exe in ("git","ffmpeg","ffprobe","node","npm"):
 print(exe, shutil.which(exe) or "não instalado (opcional/integração externa)")

## 6 — Configuração do NUBI

Configure valores padrão sem guardar segredo em arquivo do repositório. Prioriza Colab Secrets; entrada oculta é alternativa de sessão. Chaves não são impressas nem gravadas no `.env`.

In [ ]:
import os, getpass
try:
 from google.colab import userdata
 secret=userdata.get("LTX_API_KEY")
 if secret: os.environ["LTX_API_KEY"]=secret; print("Segredo LTX recebido do Colab Secrets (valor oculto).")
except Exception: pass
if not os.getenv("LTX_API_KEY") and "google.colab" in sys.modules:
 if input("Configurar chave LTX nesta sessão? [s/N]: ").strip().lower() in ("s","sim"):
  os.environ["LTX_API_KEY"]=getpass.getpass("LTX API key (não será exibida): ")
NUBI_MODE = input(f"Modo CPU/GPU/HYBRID [{NUBI_MODE}]: ").strip().upper() or NUBI_MODE
NUBI_MODE = NUBI_MODE if NUBI_MODE in {"CPU","GPU","HYBRID"} else "CPU"
NUBI_OUTPUT_DIR = pathlib.Path(os.getenv("NUBI_OUTPUT_DIR",str((NUBI_DRIVE/"outputs") if NUBI_DRIVE else NUBI_PATH/"storage"/"outputs"))).expanduser()
NUBI_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
os.environ["NUBI_OUTPUT_DIR"]=str(NUBI_OUTPUT_DIR)
os.environ["NUBI_DATA_DIR"]=str(NUBI_DRIVE if NUBI_DRIVE else NUBI_PATH/"storage")
NUBI_DEFAULTS={"quality":"balanced","resolution":"1280x720","fps":24,"language":"pt-BR","aspect_ratio":"16:9","ltx_provider":"cloud"}
print("Modo:",NUBI_MODE,"· Saída:",NUBI_OUTPUT_DIR,"· Padrões:",NUBI_DEFAULTS)
print("LTX API pode incorrer cobrança. Não foi feita geração nesta etapa.")

## 7 — Inicializar banco de dados

Inicializa SQLite no diretório persistente configurado. Nenhuma tabela de senha é utilizada pelo modo local.

In [ ]:
import sys
os.chdir(NUBI_PATH)
run_checked([sys.executable,"scripts/init_db.py"],cwd=NUBI_PATH)
db=pathlib.Path(os.environ["NUBI_DATA_DIR"])/"nubi.sqlite3"
print("Banco:",db,"· Existe:",db.exists())

## 8 — Iniciar backend e verificar API

Servidor fica vinculado a localhost. Se a inicialização falhar, a saída real do processo é exibida; não declare o serviço ativo até o health check responder.

In [ ]:
import subprocess, time, urllib.request, os, sys, pathlib
if NUBI_PROCESS and NUBI_PROCESS.poll() is None: print("Backend já está em execução.")
else:
 env=os.environ.copy();env["NUBI_HOST"]="127.0.0.1";env["NUBI_PORT"]="8000"
 NUBI_PROCESS=subprocess.Popen([sys.executable,"-m","uvicorn","backend.main:app","--host","127.0.0.1","--port","8000"],cwd=NUBI_PATH,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
 time.sleep(3)
 try:
  with urllib.request.urlopen("http://127.0.0.1:8000/api/health",timeout=5) as r: print("Health:",r.read().decode())
 except Exception as e:
  print("Falha real ao iniciar/verificar backend:",e)
  if NUBI_PROCESS.poll() is not None: print(NUBI_PROCESS.stdout.read()[-8000:])
  raise

## 9 — Abrir interface pelo proxy Colab

O proxy é temporário, limitado à sessão autenticada Colab. Não use túnel público. Fora do Colab, acesse localhost manualmente.

In [ ]:
if "google.colab" in sys.modules:
 from google.colab import output
 NUBI_URL=output.eval_js("google.colab.kernel.proxyPort(8000)")
 print("Painel NUBI (proxy desta sessão):",NUBI_URL)
else: print("Painel local: http://127.0.0.1:8000")

## 10 — Verificar serviços e engines

A detecção do binário ou endpoint não significa que o motor está renderizando. Fusion, Natron e After Effects são aplicações desktop e geralmente exigem host externo/licença; não são assumidos disponíveis no Colab.

In [ ]:
import urllib.request, json
try:
 with urllib.request.urlopen("http://127.0.0.1:8000/api/engines",timeout=10) as r: status=json.loads(r.read())
 print("Plataforma:",status["platform"])
 for e in status["engines"]: print(("DISPONÍVEL" if e["available"] else "AUSENTE"),e["name"],"—",e["detail"])
 with urllib.request.urlopen("http://127.0.0.1:8000/api/ai/ltx/status",timeout=5) as r: print("LTX:",r.read().decode())
except Exception as e: print("Não foi possível verificar os serviços:",e)
print("ComfyUI remoto exige COMFYUI_URL; configure o ambiente e valide via /api/engines.")
print("LTX autenticado só é verificado ao enviar trabalho (pode gerar cobrança).")

## 11 — Testes de backend, banco, FFmpeg e GPU

Executa testes reais do projeto e probes disponíveis. O teste LTX verifica configuração local, mas não consome créditos.

In [ ]:
import subprocess, shutil, os, sys
try:
 run_checked([sys.executable,"-m","pytest","-q"],cwd=NUBI_PATH)
except Exception as e: print("Testes automatizados falharam:",e)
for cmd in (["ffmpeg","-version"],["ffprobe","-version"]):
 if shutil.which(cmd[0]):
  p=subprocess.run(cmd,capture_output=True,text=True); print(cmd[0],"OK" if p.returncode==0 else "ERRO",(p.stdout or p.stderr).splitlines()[0] if (p.stdout or p.stderr) else "")
 else: print(cmd[0],"não disponível")
print("GPU CUDA:",bool(cuda_ok),"· armazenamento:",round(shutil.disk_usage(NUBI_OUTPUT_DIR).free/1024**3,2),"GB livres")
print("Fila de jobs: endpoints API /api/jobs e /api/summary; sem worker de render ativado nesta base.")

## 12 — Painel de controle

Consulta a API para exibir status verdadeiro; URL é temporária no Colab.

In [ ]:
import urllib.request,json
def get_json(url):
 with urllib.request.urlopen(url,timeout=5) as r:return json.loads(r.read())
print("Backend:",get_json("http://127.0.0.1:8000/api/health"))
print("Resumo:",get_json("http://127.0.0.1:8000/api/summary"))
print("GPU:",gpu_name,"· CUDA:",cuda_ok,"· modo:",NUBI_MODE)
print("LTX chave configurada:",bool(os.getenv("LTX_API_KEY")),"(autenticação ainda não verificada)")
print("Projetos e saídas:",os.environ["NUBI_DATA_DIR"],NUBI_OUTPUT_DIR)
print("Parar: execute célula de shutdown abaixo. Logs: NUBI_PROCESS.stdout.")

## 13 — Encerramento e recuperação

O SQLite mantém estado em armazenamento configurado. Jobs cloud aceitos podem ser consultados novamente pelo endpoint; Colab e URLs de proxy terminam ao encerrar a sessão. Execute as células 4 a 8 para reconectar/reiniciar após uma interrupção. Shutdown encerra servidor sem apagar arquivos.

In [ ]:
def nubi_shutdown():
 global NUBI_PROCESS
 if NUBI_PROCESS and NUBI_PROCESS.poll() is None:
  NUBI_PROCESS.terminate()
  try: NUBI_PROCESS.wait(timeout=10)
  except subprocess.TimeoutExpired: NUBI_PROCESS.kill(); NUBI_PROCESS.wait(timeout=5)
  print("Backend encerrado; dados locais preservados em",os.environ.get("NUBI_DATA_DIR"))
 else: print("Backend já estava parado.")
# Execute nubi_shutdown() quando desejar parar.

### Limitações e compatibilidade

- O notebook não baixa automaticamente modelos LTX/ComfyUI nem promete acesso a GPU.
- A API LTX cloud requer credencial e pode cobrar; duração, resolução e recursos dependem do modelo e documentação vigentes.
- O starter oferece projetos, banco, detecção real de engines, plano de cenas e submissão/polling LTX cloud. Editor multi-trilha, geração de imagens, áudio, render final e conectores desktop completos ainda não estão implementados.
- Para segurança, backend fica em localhost e a URL do proxy do Colab dura apenas na sessão. Não execute servidor sem proteção pública.